<a href="https://colab.research.google.com/github/starlton/Deep-Learning/blob/main/week2_calculus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — Numerical Differentiation

**Goal:** Understand how derivatives can be computed numerically, and why neural networks use *analytical* gradients (backpropagation) instead.

This notebook is the conceptual bridge to **micrograd** — the autograd engine we build next. Before automating gradients, we first compute them by hand, numerically.

**What we'll cover:**
1. The numerical (finite-difference) derivative
2. Testing convergence as the step size shrinks
3. The symmetric difference — a more accurate variant
4. Why analytical gradients beat numerical ones for real networks

---

## Setup — Imports

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

%matplotlib inline

---
## 1. The Numerical Derivative

The derivative of a function measures its **instantaneous rate of change** — the slope of the tangent line at a point.

The formal definition of the derivative is:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

We can't take a true limit on a computer, but we can **approximate** it by using a small (but nonzero) step size $h$:

$$f'(x) \approx \frac{f(x+h) - f(x)}{h}$$

This is called the **forward difference** (or one-sided difference). The smaller $h$ is, the closer the approximation — up to a point (very small $h$ introduces floating-point rounding errors).

### Test case
We test on $f(x) = x^2$. By the power rule, its exact derivative is:

$$f'(x) = 2x \quad \Rightarrow \quad f'(3) = 6$$

So our numerical estimate should converge to **6** as $h$ shrinks.

In [ ]:
h = 0.001 # h val

def f(x):
  return x**2

x = 3.0

def derivative(x, h_vals):
  results = []
  for h in h_vals:
    f_prime = (f(x+h) - f(x)) / h
    results.append(f_prime)

  return results


h_vals = [0.01, 0.001, 0.0001, 0.00001]

derivative(x, h_vals)

### Observation

| $h$ | Forward difference estimate | Error from 6 |
|---|---|---|
| 0.01 | 6.0100 | 0.0100 |
| 0.001 | 6.0010 | 0.0010 |
| 0.0001 | 6.0001 | 0.0001 |
| 0.00001 | 6.00001 | 0.00001 |

**The error shrinks proportionally to $h$.** Halving $h$ roughly halves the error. This is called **first-order accuracy** — written as $O(h)$.

---
## 2. The Symmetric Difference

We can do much better by sampling the function on **both sides** of $x$ instead of just one:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

This is the **central difference** (or symmetric difference). It is more accurate because the errors on either side partially cancel out.

Where the forward difference has error proportional to $h$, the symmetric difference has error proportional to $h^2$ — written $O(h^2)$. Since $h$ is already small, $h^2$ is dramatically smaller.

In [ ]:
def f(x):
  return x**2

h = 0.01
x = 3.0

sym_prime = (f(x+h) - f(x-h)) / (2*h)

sym_prime

def sym_deriv(x, h_vals):
  results = []
  for h in h_vals:
    f_prime = (f(x+h) - f(x-h)) / (2*h)
    results.append(f_prime)

  return results

sym_deriv(x, h_vals)

In [ ]:
np.allclose(sym_deriv(x, h_vals), 6) # Practice with .allclose()

### Forward vs Symmetric — accuracy comparison at $h = 0.01$

| Method | Formula | Estimate at $x=3$ | Error | Accuracy |
|---|---|---|---|---|
| Forward difference | $\frac{f(x+h) - f(x)}{h}$ | 6.0100 | ~$10^{-2}$ | $O(h)$ |
| Symmetric difference | $\frac{f(x+h) - f(x-h)}{2h}$ | 5.99999... | ~$10^{-10}$ | $O(h^2)$ |

**Conclusion:** for the same step size $h$, the symmetric difference is far more accurate — eight orders of magnitude better in this example. This is why central differences are the standard choice for numerical gradient checking.

---
## 3. Why Analytical Gradients Beat Numerical Ones

The numerical gradient *approximates* the slope by nudging $x$ by a tiny $h$ and measuring the change. **Backpropagation** (what micrograd's `.backward()` does) computes the *exact* derivative analytically using the **chain rule**.

Both estimate the same quantity — the slope — but numerical approximates while analytical is exact.

### The chain rule (the heart of backprop)

If $y = f(g(x))$, then:

$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$$

Backprop applies this rule repeatedly, propagating gradients backward from the output through every operation in the computational graph to each input.

### Why analytical wins for real networks

**1. Speed.** Numerical differentiation requires a separate forward pass for *every single parameter*. A network with 1,000,000 parameters would need over a million forward passes to estimate one gradient step. Backprop computes **all** gradients in a single backward pass.

**2. Accuracy.** The numerical estimate always carries approximation error from finite $h$, and choosing $h$ is a tradeoff:
- $h$ too large → poor approximation of the limit
- $h$ too small → floating-point rounding errors dominate

Analytical gradients have no such error — they are exact.

### So why learn the numerical version at all?

Because it's the perfect **debugging tool**. When you build micrograd, you can verify its analytical gradients are correct by comparing them against the numerical estimate. If they match, your backprop implementation is correct. This is called **gradient checking**.

---
## Summary

| Concept | Formula | Key point |
|---|---|---|
| Derivative (definition) | $\lim_{h\to0}\frac{f(x+h)-f(x)}{h}$ | Instantaneous rate of change |
| Forward difference | $\frac{f(x+h)-f(x)}{h}$ | Simple, error $O(h)$ |
| Symmetric difference | $\frac{f(x+h)-f(x-h)}{2h}$ | More accurate, error $O(h^2)$ |
| Chain rule | $\frac{dy}{dx}=\frac{dy}{dg}\cdot\frac{dg}{dx}$ | The basis of backpropagation |

**Next: Week 2 Day 3 — build micrograd, our own autograd engine that computes these gradients analytically.**